# 02. Data Preparation & Preprocessing Pipeline

**Judul Proyek:** Deteksi Dini Risiko Kecemasan Sedang–Berat pada Gamer Online Berdasarkan Pola Bermain dan Faktor Psikologis Menggunakan Pendekatan Multi-Model

**Kelompok:** Kelompok 2 / TRPL 5B
- Mohammad Hanif Huda A (NIM: 244311049)
- Muhammad Akbar Fadilah (NIM: 244311051)

**Fokus Lifecycle:** **[L] Data Preparation (Bab 5)**

**Tujuan:** Membangun *data pipeline* yang bersih, bebas dari kebocoran informasi (*data leakage*), melakukan partisi data stratified, penskalaan fitur numerik, *encoding* fitur kategorikal, dan penyiapan bobot kelas untuk pemodelan (Bab 6).

## 1. Import Library & Konfigurasi Reproduktibilitas

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Reproducibility Seed ditetapkan: {RANDOM_SEED}")

Reproducibility Seed ditetapkan: 42


## 2. Memuat Dataset & Pembersihan Awal (Bab 5.1)

In [2]:
data_path = '../data/GamingStudy_data.csv' if os.path.exists('../data/GamingStudy_data.csv') else 'data/GamingStudy_data.csv'
df_raw = pd.read_csv(data_path, encoding='latin1')

# 1. Kriteria Inklusi: Responden berstatus 'Accept'
df = df_raw[df_raw['accept'] == 'Accept'].copy()
print(f"Data tersaring (Accept): {df.shape[0]} baris x {df.shape[1]} kolom")

# 2. Penanganan Outlier pada Jam Bermain (Hours <= 84 jam/minggu)
df = df[(df['Hours'] <= 84) & (df['Hours'].notnull())].copy()
print(f"Data setelah filter outlier jam bermain (<= 84): {df.shape[0]} baris")

Data tersaring (Accept): 13050 baris x 55 kolom
Data setelah filter outlier jam bermain (<= 84): 12981 baris


## 3. Formulasi Target & Eksklusi Kebocoran Label (*Label Leakage Exclusion*) (Bab 5.1 - 5.2)

In [3]:
# Binarisasi target: 1 jika GAD_T >= 10 (Sedang-Berat), 0 jika < 10
y = (df['GAD_T'] >= 10).astype(int).values

# Identifikasi kolom bocor dan kolom yang tidak informatif
leakage_cols = ['GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'GAD_T', 'GADE']
uninformative_cols = ['S. No.', 'Timestamp', 'accept', 'highestleague', 'League', 
                      'Birthplace', 'Residence', 'Reference', 'Residence_ISO3', 'Birthplace_ISO3']

drop_cols = leakage_cols + uninformative_cols
features_df = df.drop(columns=[col for col in drop_cols if col in df.columns]).copy()

print(f"Matriks Fitur (X) Tersisa: {features_df.shape[1]} fitur kandidat")
print("Daftar Fitur:", list(features_df.columns))

Matriks Fitur (X) Tersisa: 36 fitur kandidat
Daftar Fitur: ['SWL1', 'SWL2', 'SWL3', 'SWL4', 'SWL5', 'Game', 'Platform', 'Hours', 'earnings', 'whyplay', 'streams', 'SPIN1', 'SPIN2', 'SPIN3', 'SPIN4', 'SPIN5', 'SPIN6', 'SPIN7', 'SPIN8', 'SPIN9', 'SPIN10', 'SPIN11', 'SPIN12', 'SPIN13', 'SPIN14', 'SPIN15', 'SPIN16', 'SPIN17', 'Narcissism', 'Gender', 'Age', 'Work', 'Degree', 'Playstyle', 'SWL_T', 'SPIN_T']


## 4. Feature Engineering: Pengelompokan Kategori Berfrekuensi Rendah (Bab 5.2)

In [4]:
# 1. Ringkas variabel 'Game' (hanya pertahankan top 5, sisanya 'Other')
top_games = ['League of Legends', 'Counter Strike', 'Starcraft 2', 'World of Warcraft', 'Other']
features_df['Game_Clean'] = features_df['Game'].apply(lambda x: x if x in top_games else 'Other')

# 2. Ringkas 'Playstyle'
top_styles = [
    'Multiplayer - online - with real life friends',
    'Multiplayer - online - with strangers',
    'Multiplayer - online - with online acquaintances or teammates',
    'Singleplayer'
]
features_df['Playstyle_Clean'] = features_df['Playstyle'].apply(lambda x: x if x in top_styles else 'Other')

# 3. Ringkas 'whyplay'
top_why = ['having fun', 'improving', 'winning', 'relaxing']
features_df['whyplay_Clean'] = features_df['whyplay'].apply(lambda x: x if x in top_why else 'Other')

# 4. Fitur Interaksi Psikologis: Rasio Fobia Sosial terhadap Kepuasan Hidup
features_df['SPIN_SWL_Ratio'] = features_df['SPIN_T'] / (features_df['SWL_T'] + 1.0)

# Hapus kolom teks mentah lama
features_df = features_df.drop(columns=['Game', 'Playstyle', 'whyplay'], errors='ignore')
print("Fitur pasca-engineering:", list(features_df.columns))

Fitur pasca-engineering: ['SWL1', 'SWL2', 'SWL3', 'SWL4', 'SWL5', 'Platform', 'Hours', 'earnings', 'streams', 'SPIN1', 'SPIN2', 'SPIN3', 'SPIN4', 'SPIN5', 'SPIN6', 'SPIN7', 'SPIN8', 'SPIN9', 'SPIN10', 'SPIN11', 'SPIN12', 'SPIN13', 'SPIN14', 'SPIN15', 'SPIN16', 'SPIN17', 'Narcissism', 'Gender', 'Age', 'Work', 'Degree', 'SWL_T', 'SPIN_T', 'Game_Clean', 'Playstyle_Clean', 'whyplay_Clean', 'SPIN_SWL_Ratio']


## 5. Partisi Data: Stratified Train-Test Split (Bab 5.4)

In [5]:
# Membagi data menjadi 80% Training Set dan 20% Hold-out Test Set
X_train, X_test, y_train, y_test = train_test_split(
    features_df, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print(f"Dimensi Training Set : {X_train.shape[0]} sampel")
print(f"Dimensi Test Set     : {X_test.shape[0]} sampel")
print(f"Proporsi Positif Train: {np.mean(y_train)*100:.2f}%")
print(f"Proporsi Positif Test : {np.mean(y_test)*100:.2f}%")

Dimensi Training Set : 10384 sampel
Dimensi Test Set     : 2597 sampel
Proporsi Positif Train: 17.38%
Proporsi Positif Test : 17.40%


## 6. Pipeline Transformasi: Imputasi, One-Hot Encoding & Penskalaan (Bab 5.3)

In [6]:
# Identifikasi kolom numerik dan kategorikal
numeric_cols = ['Hours', 'streams', 'Age', 'SWL_T', 'SPIN_T', 'Narcissism', 'SPIN_SWL_Ratio']
categorical_cols = ['Platform', 'Gender', 'Work', 'Degree', 'Game_Clean', 'Playstyle_Clean', 'whyplay_Clean']

# Pipeline Numerik: Imputasi Median -> RobustScaler
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Pipeline Kategorikal: Imputasi Modus -> OneHotEncoder
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Gabungkan ke dalam ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numeric_cols),
    ('cat', cat_transformer, categorical_cols)
])

# PENTING: Fit HANYA pada data latih (X_train) untuk mencegah data leakage!
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Ekstraksi nama fitur hasil OneHotEncoding
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_cols)
all_feature_names = numeric_cols + list(cat_feature_names)

print(f"Dimensi Matriks Fitur Siap Latih (X_train_proc): {X_train_proc.shape}")
print(f"Dimensi Matriks Fitur Uji (X_test_proc)        : {X_test_proc.shape}")
print(f"Total Fitur Akhir: {len(all_feature_names)} dimensi")

Dimensi Matriks Fitur Siap Latih (X_train_proc): (10384, 36)
Dimensi Matriks Fitur Uji (X_test_proc)        : (2597, 36)
Total Fitur Akhir: 36 dimensi


## 7. Data Balancing & Perhitungan Class Weights (Bab 5.5)

In [7]:
# Perhitungan Cost-Sensitive Learning Class Weights pada Data Training
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {cls: weight for cls, weight in zip(classes, weights)}

print("=== Bobot Penyeimbang Kelas (Cost-Sensitive Learning) ===")
print(f"Bobot Kelas 0 (Risiko Rendah): {class_weight_dict[0]:.4f}")
print(f"Bobot Kelas 1 (Risiko Tinggi): {class_weight_dict[1]:.4f}")

# Opsional: Demonstrasi SMOTE (HANYA diterapkan pada data latih)
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=RANDOM_SEED)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_proc, y_train)
    print(f"\nSetelah SMOTE pada Training Set: {X_train_smote.shape[0]} sampel (Distribusi seimbang: {np.bincount(y_train_smote)})")
except ImportError:
    print("\nLibrary imblearn belum terpasang, pemodelan akan memanfaatkan penalti loss berbobot (class_weight='balanced')")

=== Bobot Penyeimbang Kelas (Cost-Sensitive Learning) ===
Bobot Kelas 0 (Risiko Rendah): 0.6052
Bobot Kelas 1 (Risiko Tinggi): 2.8765

Library imblearn belum terpasang, pemodelan akan memanfaatkan penalti loss berbobot (class_weight='balanced')


## 8. Verifikasi Akhir Integritas & Kesiapan Data

Ringkasan Integritas Data Preparation:
1. **Bebas Data Leakage**: Seluruh parameter penskalaan (`median`, `IQR`) dan tabel *One-Hot Encoding* dipelajari murni dari `X_train`.
2. **Bebas Label Leakage**: Variabel kuesioner GAD (`GAD1`–`GAD7`, `GAD_T`, `GADE`) telah dieksklusi sepenuhnya.
3. **Zero NaN**: Tidak ada *missing values* yang tersisa pada matriks masukan `X_train_proc` dan `X_test_proc`.
4. **Class Imbalance Ready**: Tersedia kamus bobot kelas terhitung untuk dioperasikan pada **Logistic Regression**, **Random Forest**, dan fungsi loss *Weighted Binary Cross-Entropy* pada **ResNet** (Bab 6).